[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/18_Deduplication.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# Semantica 去重（Deduplication）

欢迎来到**去重**走读！知识图谱的数据常常来自多个来源，重复实体在所难免（比如 "Apple Inc." 和 "Apple Inc"）。

Semantica 提供坚实的**去重模块**帮你：
1. **算相似度**：用字符串、属性和嵌入比较实体
2. **查重复**：找出代表同一现实对象的实体对或实体组
3. **聚簇**：把相似实体归堆
4. **合并**：把重复实体合成一个规范实体，同时解决冲突

### 先看主线

与 17 篇是姊妹篇：17 篇解决"值打架"（同一实体、不同说法），本篇解决"名不同"（同一实体、多条记录）。合起来正是 Semantica 工程纪律的完整两步——**先标记冲突、再去重合并，数据才能进图谱**。07 篇第 4 步用过的 `DuplicateDetector`、`EntityMerger` 就出自这个模块，这里展开讲透。

本篇四问递进：

**像不像（第 2 节）—— 相似度**

`SimilarityCalculator` 给两个实体打 0 到 1 的分：字符串、属性、关系、嵌入各有权重——权重就是你的领域知识。

**是不是（第 3 节）—— 检测**

阈值过线算重复。配对之外还有增量检测：新数据来了不必全库重比。

**归哪堆（第 4 节）—— 聚类**

A 像 B、B 像 C，但 A 不一定像 C——链式相似靠配对抓不全，聚类才能归堆。

**合成谁（第 5–6 节）—— 合并**

五种合并策略选代表；完整工作流 = 合并结果 + 没进任何堆的单身实体（07 篇的易错点，这里用代码再走一遍）。

In [ ]:
# Install Semantica
!pip install -q semantica

## 1. 准备样本数据

造一批刻意带重复的数据，模拟多来源（比如一个 CRM 加一个公共数据库）。

**实体阵容：**
- **Apple**：变体 "Apple Inc."、"Apple Inc"、裸名 "Apple"
- **Microsoft**：变体 "Microsoft Corp" 和 "Microsoft"
- **Google**：唯一实体，做对照组

In [ ]:
entities = [
    # Apple Variations
    {
        "id": "e1",
        "name": "Apple Inc.",
        "type": "Company",
        "properties": {"industry": "Technology", "hq": "Cupertino", "founded": 1976},
        "relationships": [{"predicate": "founded_by", "object": "Steve Jobs"}]
    },
    {
        "id": "e2",
        "name": "Apple Inc",
        "type": "Company",
        "properties": {"industry": "Tech", "hq": "Cupertino, CA"}, # Slightly different properties
        "relationships": []
    },
    {
        "id": "e3",
        "name": "Apple",
        "type": "Company",
        "properties": {"industry": "Consumer Electronics"}, 
        "relationships": [{"predicate": "ceo", "object": "Tim Cook"}]
    },
    
    # Microsoft Variations
    {
        "id": "e4",
        "name": "Microsoft Corp",
        "type": "Company",
        "properties": {"industry": "Software", "hq": "Redmond"}
    },
    {
        "id": "e5",
        "name": "Microsoft",
        "type": "Company",
        "properties": {"industry": "Tech", "hq": "Redmond, WA"}
    },
    
    # Unique Entity
    {
        "id": "e6",
        "name": "Google LLC",
        "type": "Company",
        "properties": {"industry": "Internet"}
    }
]

print(f"Created {len(entities)} sample entities.")

## 2. 相似度计算

`SimilarityCalculator` 是核心引擎：比较两个实体，返回 0 到 1 的分数。它看四个方面：
- **字符串相似度**：名称和文本字段
- **属性相似度**：键值对的重合程度
- **关系相似度**：与其他实体的连接
- **嵌入**：语义向量相似度（如果可用）

每个因子的权重都可以自定义。

In [ ]:
from semantica.deduplication import SimilarityCalculator, SimilarityResult
# Initialize calculator with custom weights
calculator = SimilarityCalculator(
    string_weight=0.5,      # High importance on name
    property_weight=0.3,    # Medium importance on properties
    relationship_weight=0.2 # Lower importance on relationships
)

# Compare "Apple Inc." (e1) vs "Apple Inc" (e2)
score_e1_e2 = calculator.calculate_similarity(entities[0], entities[1])

print(f"Similarity between '{entities[0]['name']}' and '{entities[1]['name']}':")
print(f"  Total Score: {score_e1_e2.score:.4f}")
print(f"  Breakdown:   {score_e1_e2.components}")

# Compare "Apple Inc." (e1) vs "Microsoft" (e5)
score_e1_e5 = calculator.calculate_similarity(entities[0], entities[4])

print(f"\nSimilarity between '{entities[0]['name']}' and '{entities[4]['name']}':")
print(f"  Total Score: {score_e1_e5.score:.4f}")

## 3. 重复检测

`DuplicateDetector` 用相似度计算器扫描数据集找重复。它能找：
- **对**：简单的 A 匹配 B
- **组**：A 匹配 B，B 又匹配 C

`similarity_threshold` 决定多大分算匹配。

In [ ]:
# Import specific classes for Duplicate Detection
from semantica.deduplication import DuplicateDetector, DuplicateCandidate, DuplicateGroup
from semantica.deduplication import DeduplicationConfig

detector = DuplicateDetector(
    similarity_threshold=0.7, 
    confidence_threshold=0.6  
)

# Detect pairs
candidates = detector.detect_duplicates(entities)

print(f"Found {len(candidates)} duplicate pairs:")
for c in candidates:
    print(f"  - {c.entity1['name']} <==> {c.entity2['name']} (Score: {c.similarity_score:.2f})")

### 增量检测
已有数据库、又来了新数据？不想全库重比，用 `incremental_detect`——只比新数据和现有库。

In [ ]:
existing_db = entities[:3] # The Apple entities
new_data = [entities[4]]   # Microsoft
# Check if new data matches anything in existing DB
inc_candidates = detector.incremental_detect(new_data, existing_db)

print(f"New matches found: {len(inc_candidates)}")
# Expected: 0, because Microsoft is not Apple.

## 4. 聚类

配对有时不够用。`ClusterBuilder` 把相互关联的实体聚成簇——这样才能看清一个重复实体的完整范围。

In [ ]:
# Import specific classes for Clustering
from semantica.deduplication import ClusterBuilder, Cluster, ClusterResult
cluster_builder = ClusterBuilder(threshold=0.7)
result = cluster_builder.build_clusters(entities)

print(f"Found {len(result.clusters)} clusters:")
for i, cluster in enumerate(result.clusters):
    names = [e['name'] for e in cluster.entities]
    print(f"  Cluster {i+1}: {names}")

cluster_builder = ClusterBuilder(threshold=0.7)
result = cluster_builder.build_clusters(entities)

print(f"Found {len(result.clusters)} clusters:")
for i, cluster in enumerate(result.clusters):
    names = [e['name'] for e in cluster.entities]
    print(f"  Cluster {i+1}: {names}")

## 5. 实体合并

重复找到了，`EntityMerger` 负责合并。要选一种**合并策略**：

- `KEEP_FIRST` / `KEEP_LAST`：按顺序取
- `KEEP_MOST_COMPLETE`：保留数据最全的实体（属性 + 关系）
- `KEEP_HIGHEST_CONFIDENCE`：按内部置信度取
- `MERGE_ALL`：全部合体（数组拼接、冲突投票解决）

In [ ]:
# Import specific classes for Entity Merging
from semantica.deduplication import EntityMerger, MergeStrategy, MergeStrategyManager, MergeOperation, MergeResult

In [ ]:
merger = EntityMerger()

# We will use the 'KEEP_MOST_COMPLETE' strategy
# This ensures we don't lose valuable information from richer entities
merge_ops = merger.merge_duplicates(
    entities, 
    strategy=MergeStrategy.KEEP_MOST_COMPLETE
)

print(f"Performed {len(merge_ops)} merge operations.")

print("\n--- Merged Results ---")
for op in merge_ops:
    final_ent = op.merged_entity
    original_count = len(op.source_entities)
    print(f"Merged {original_count} entities into: '{final_ent['name']}'")
    print(f"  - Final Properties: {final_ent['properties']}")
    print(f"  - Final Relationships: {len(final_ent.get('relationships', []))}")

## 6. 完整工作流

收个尾：包成一个干净的函数——脏数据进、干净数据出。注意函数里"合并结果 + 单身实体"两段拼合——这是完整去重集的正确姿势（07 篇第 4 步踩过的坑）。

In [ ]:
def deduplicate_dataset(raw_entities):
    print("1. Detecting duplicates...")
    # Step 1: Detect
    detector = DuplicateDetector(similarity_threshold=0.75)
    # We can skip explicit detection calls if we just want to merge, 
    # as EntityMerger calls detection internally, but doing it manually allows inspection.
    
    print("2. Merging entities...")
    # Step 2: Merge
    merger = EntityMerger()
    ops = merger.merge_duplicates(raw_entities, strategy=MergeStrategy.KEEP_MOST_COMPLETE)
    
    # Let's collect all final IDs to see what remains
    merged_entities = [op.merged_entity for op in ops]
    
    # Find entities that were NOT part of any merge (singletons)
    merged_ids = set()
    for op in ops:
        for source in op.source_entities:
            merged_ids.add(source['id'])
            
    singletons = [e for e in raw_entities if e['id'] not in merged_ids]
    
    final_dataset = merged_entities + singletons
    return final_dataset

# Run the workflow
clean_data = deduplicate_dataset(entities)

print(f"\nOriginal Size: {len(entities)}")
print(f"Cleaned Size:  {len(clean_data)}")
print("\nFinal Entity Names:")
for e in clean_data:
    print(f"  - {e['name']}")

## 小结

### 回顾主线

四问走完：**像不像**（加权相似度）、**是不是**（阈值检测 + 增量）、**归哪堆**（聚类对付链式相似）、**合成谁**（五种策略选代表）。与 17 篇合璧，"先冲突标记、再去重合并、才进图谱"的纪律闭环完成——多源脏数据从此能干净入库。

### 你学会了

1. **导入**去重所需的各个类
2. **计算**实体间相似度
3. **检测**重复（阈值可配置）
4. **聚类**相似实体
5. **合并**重复项，得到干净的规范数据集

维护高质量知识图谱，这个模块必不可少——尤其是从多个可能很脏的来源摄取数据时。